# Morris-Lecar: a neuron with a Hopf onset

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/09_morris_lecar.ipynb)

The Morris-Lecar model of a barnacle muscle fibre, in Rinzel and Ermentrout's
parameters: calcium current with instant activation, potassium current with recovery
`w`. Here repetitive firing starts at a Hopf bifurcation as the applied current `i` grows.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.3.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("morris_lecar")
      .comment("Morris-Lecar neuron, Hopf regime (Rinzel & Ermentrout)")
      .function("minf(v)", ".5*(1+tanh((v-v1)/v2))")
      .function("winf(v)", ".5*(1+tanh((v-v3)/v4))")
      .function("tauw(v)", "1/cosh((v-v3)/(2*v4))")
      .ode("v", "(i+gl*(vl-v)+gk*w*(vk-v)+gca*minf(v)*(vca-v))/c")
      .ode("w", "phi*(winf(v)-w)/tauw(v)")
      .aux("ica", "gca*minf(v)*(v-vca)")
      .parameter(i=60, c=20, vk=-84, vl=-60, vca=120, gk=8, gl=2, gca=4.4)
      .parameter(v1=-1.2, v2=18, v3=2, v4=30, phi=0.04)
      .initial(v=-60, w=0.015)
      .option(total=400, dt=0.25, meth="rk4", bound=10000))
print(mb.render())

# Morris-Lecar neuron, Hopf regime (Rinzel & Ermentrout)
minf(v)=.5*(1+tanh((v-v1)/v2))
winf(v)=.5*(1+tanh((v-v3)/v4))
tauw(v)=1/cosh((v-v3)/(2*v4))
v'=(i+gl*(vl-v)+gk*w*(vk-v)+gca*minf(v)*(vca-v))/c
w'=phi*(winf(v)-w)/tauw(v)
aux ica=gca*minf(v)*(v-vca)
par i=60,c=20,vk=-84,vl=-60,vca=120,gk=8,gl=2,gca=4.4
par v1=-1.2,v2=18,v3=2,v4=30,phi=0.04
init v=-60,w=0.015
@ total=400,dt=0.25,meth=rungekutta,bound=10000
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="v", y="w", window=(-75, 45, -0.1, 0.55),
              sliders={"i": (0, 300), "phi": (0.01, 0.12), "gca": (2, 8)},
              auto="i", auto_range=(0, 300), auto2="phi", auto2_range=(0.01, 0.4), title="Morris-Lecar")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Firing rate against current** (an f-I curve) from simulations, next to the Hopf point
from AUTO.

In [5]:
import numpy as np
I = np.linspace(60, 250, 25)
rate = []
for i_ in I:
    tr = ex.m.run(keep=False, i=i_, total=1500, trans=500)
    v = tr["v"]; up = np.where((v[:-1] < 0) & (v[1:] >= 0))[0]
    rate.append(1000 * len(up) / 1000)        # spikes per second (t in ms)
parts = ex.run_auto()
hb = sorted(p.x for d in parts for p in d.special("HB"))
fig = go.Figure(go.Scatter(x=I, y=rate, mode="lines+markers", name="simulation"))
for h in hb:
    fig.add_vline(x=h, line_dash="dot", annotation_text=f"HB {h:.1f}")
show_figure(fig, xaxis_title="i", yaxis_title="spikes / s", height=320)

## Things to try

- Pick the HB point in the AUTO panel and compute its periodic orbits: are they born stable?
- Change `phi` (the speed of recovery) and watch the Hopf point move.